# `config.ipynb` - Configuración del proyecto GeoKW

Módulo de configuración: parámetros técnicos, rutas, reglas de negocio y la **tabla maestra socioeconómica** (data seeding) del proyecto GeoKW - Índice de Oportunidad de Inversión (IOI) para la instalación de puntos de recarga de vehículo eléctrico en Sevilla.

> Se ejecuta con `%run geokw_etl/config.ipynb` desde `main.ipynb`. Todas las constantes definidas aquí quedan disponibles para el resto de módulos.

In [ ]:
import os

## 1. Sistemas de referencia y geometría

- `CRS_GEOGRAFICO` (EPSG:4326): el que usan el GeoJSON de distritos y la API de puntos de recarga (lat/lon en grados).
- `CRS_PROYECTADO` (EPSG:25830, ETRS89 / UTM zona 30N): sistema oficial para España peninsular, en metros - se usa para calcular áreas y distancias con precisión (p. ej. el radio de 20 km alrededor del centro de Sevilla).

In [ ]:
CRS_GEOGRAFICO = "EPSG:4326"
CRS_PROYECTADO = "EPSG:25830"

# Centro de Sevilla (Plaza Nueva), usado como origen del radio de 20 km
# para la consulta a la API de puntos de recarga
SEVILLA_CENTRO_LAT = 37.3900
SEVILLA_CENTRO_LON = -5.9961
RADIO_KM = 20

## 2. Rutas de entrada y salida

In [ ]:
RUTA_GEOJSON_DISTRITOS = os.environ.get("GEOKW_RUTA_DISTRITOS", "./data/distritos_sevilla.geojson")
RUTA_SALIDA_MAPA = os.environ.get("GEOKW_RUTA_MAPA", "./output/mapa_sevilla.html")
RUTA_LOG = os.environ.get("GEOKW_RUTA_LOG", "./logs/ejecucion_etl.log")

## 3. API REST de puntos de recarga (Open Charge Map)

Endpoint público de [Open Charge Map](https://openchargemap.org/). Con `API_MAX_RESULTADOS > 250` (como aquí, 1000), Open Charge Map **exige** una clave - gratuita, en [openchargemap.org/site/profile/applications](https://openchargemap.org/site/profile/applications). La clave por defecto de abajo es la clave real de este proyecto (app "Cargadores Sevilla"); para usar la tuya, sobreescríbela con la variable de entorno `OCM_API_KEY` o cambia el valor por defecto directamente aquí.

`API_MAX_REINTENTOS` controla cuántas veces se reintenta antes de activar el dataset de respaldo interno (fallback) - ver `extract_cargadores.ipynb`.

> **Nota sobre el `User-Agent`**: Open Charge Map devuelve `403 Forbidden` a los User-Agent genéricos de librería (p. ej. `python-requests/2.31.0`), por considerarlos tráfico de robot - está documentado en su propio foro de comunidad. `USER_AGENT_API` declara un User-Agent identificable para evitarlo.

In [ ]:
API_OPENCHARGEMAP_URL = "https://api.openchargemap.io/v3/poi/"
API_KEY_OPENCHARGEMAP = os.environ.get("OCM_API_KEY", "b42f5cc2-f5b2-43a2-b17d-e4b85ea166cb")
API_TIMEOUT_SEGUNDOS = 10
API_MAX_REINTENTOS = 2
API_MAX_RESULTADOS = 1000

# Open Charge Map banea con 403 los User-Agent genericos de librerias
# (python-requests/x.x se identifica como "robot"). Se declara un
# User-Agent propio, identificando la app registrada en openchargemap.org.
USER_AGENT_API = "GeoKW-Sevilla-IOI-Pipeline/1.0 (+https://openchargemap.org/site/profile/applications)"

## 4. Constantes de las fórmulas (KPIs)

- `FACTOR_DENSIDAD`: constante ×10.000 de la fórmula de Densidad Energética por Habitante.
- `factor de amortiguacion` no aplica aqui (eso solo se uso para la tabla de respaldo, ver seccion 6).


In [ ]:
FACTOR_DENSIDAD = 10_000  # Densidad = (Potencia Total / Poblacion) * 10.000

## 5. Diseño del visor cartográfico

Parámetros exigidos por la especificación de diseño UX/UI (sección 5 del documento de requisitos).

In [ ]:
TILES_MAPA_BASE = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
TILES_ATRIBUCION = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"
LEYENDA_TITULO = "Índice Oportunidad (IOI)"
LEYENDA_ANCHO_PX = 240
ZOOM_INICIAL = 11

## 6. Tabla maestra socioeconómica (Data Seeding INE / Ayuntamiento)

Requisito: *"Inserción determinista de la renta media neta por persona (€) procedente del Atlas de Distribución de la Renta de los Hogares (INE) y la población empadronada por distrito (Ayuntamiento de Sevilla)."*

### Población - Ayuntamiento de Sevilla, Padrón Municipal a 1/1/2024

Fuente: *Informe Socioeconómico de la Ciudad de Sevilla 2023* (Servicio de Estadística, Ayuntamiento de Sevilla). Los 11 valores suman exactamente 697.233 habitantes, coincidiendo con el total oficial publicado - dato verificado, no estimado.

### Renta neta media por persona - INE, Atlas de Distribución de Renta de los Hogares, año 2023

Fuente: tabla `31205` del INE (*Indicadores de renta media y mediana*, provincia de Sevilla), filtrada a los 11 distritos del municipio de Sevilla capital (códigos `41091 01`–`41091 11`). Valores reales extraídos y verificados directamente del fichero oficial - no son una estimación.

In [ ]:
# Población empadronada por distrito (habitantes, 1/1/2024)
# Fuente: Ayuntamiento de Sevilla, Servicio de Estadistica
POBLACION_DISTRITOS = {
    "CASCO ANTIGUO": 56_980,
    "MACARENA": 76_059,
    "NERVION": 51_276,
    "CERRO-AMATE": 90_913,
    "SUR": 69_359,
    "TRIANA": 47_114,
    "NORTE": 70_963,
    "SAN PABLO-SANTA JUSTA": 59_018,
    "ESTE-ALCOSA-TORREBLANCA": 107_133,
    "BELLAVISTA-LA PALMERA": 42_647,
    "LOS REMEDIOS": 25_771,
}
assert sum(POBLACION_DISTRITOS.values()) == 697_233, "el total no coincide con el padron oficial"

# Renta neta media por persona (EUR, ano 2023)
# Fuente: INE, Atlas de Distribucion de Renta de los Hogares, tabla 31205
RENTA_DISTRITOS = {
    "CASCO ANTIGUO": 20_777,
    "MACARENA": 12_576,
    "NERVION": 20_984,
    "CERRO-AMATE": 9_993,
    "SUR": 15_297,
    "TRIANA": 17_262,
    "NORTE": 12_100,
    "SAN PABLO-SANTA JUSTA": 15_838,
    "ESTE-ALCOSA-TORREBLANCA": 13_175,
    "BELLAVISTA-LA PALMERA": 16_488,
    "LOS REMEDIOS": 21_423,
}

print(f"Tabla maestra cargada: {len(POBLACION_DISTRITOS)} distritos, poblacion total {sum(POBLACION_DISTRITOS.values()):,} hab.".replace(",", "."))

## 7. Dataset de respaldo (fallback) de puntos de recarga

Requisito de resiliencia: si la API de Open Charge Map no está disponible, el pipeline debe seguir funcionando con un mecanismo secundario interno.

Sin API, Sevilla cuenta con del orden de **20-30 estaciones** de recarga conocidas; con la API en vivo (radio de 20 km del centro) se obtienen del orden de **350 puntos**. El fallback replica el escenario "sin API": una selección real de puntos de recarga municipales, hoteleros y comerciales de Sevilla.

> **Fuente**: recopilación de ubicaciones reales de electrolineras en Sevilla (Grupo Concesur / Clúster TLVE), con coordenadas aproximadas por geocodificación manual de cada dirección. Incluye, a propósito, algunos puntos que quedan **fuera de los 11 distritos oficiales** (p. ej. en municipios colindantes como San Juan de Aznalfarache o Tomares, que caen dentro del radio de 20 km pero no pertenecen al término municipal de Sevilla) - esto permite comprobar que el *spatial join* los desvía correctamente a cuarentena, igual que ocurre con la potencia instalada en la Isla de la Cartuja, que administrativamente no pertenece a ninguno de los 11 distritos.

In [ ]:
# Dataset de respaldo: puntos de recarga reales conocidos en Sevilla y su
# area metropolitana cercana (potencia en kW estimada segun tipo de punto:
# Schuko/lento ~3.7 kW, Tipo 2/semirrapido ~22 kW, DC rapido ~50 kW)
ESTACIONES_FALLBACK = [
    {"nombre": "Emasesa - Sede Central (C/ Escuelas Pias)",        "lat": 37.4010, "lon": -5.9950, "potencia_kw": 22},
    {"nombre": "Emasesa - EDAR San Jeronimo",                       "lat": 37.4350, "lon": -5.9700, "potencia_kw": 22},
    {"nombre": "Lipasam - Parque Central (C/ Virgen de la Oliva)",  "lat": 37.4180, "lon": -5.9850, "potencia_kw": 22},
    {"nombre": "Lipasam - Centro Pino Montano",                     "lat": 37.4330, "lon": -5.9600, "potencia_kw": 22},
    {"nombre": "Parque Movil del Ayuntamiento",                     "lat": 37.3650, "lon": -5.9850, "potencia_kw": 22},
    {"nombre": "AP Arenal",                                         "lat": 37.3850, "lon": -6.0010, "potencia_kw": 22},
    {"nombre": "AP Jose Laguillo",                                  "lat": 37.3870, "lon": -5.9800, "potencia_kw": 22},
    {"nombre": "AP Mercado de Triana",                              "lat": 37.3870, "lon": -6.0030, "potencia_kw": 22},
    {"nombre": "AP Avda. de Roma",                                  "lat": 37.3800, "lon": -5.9950, "potencia_kw": 22},
    {"nombre": "AP Plaza de Cuba",                                  "lat": 37.3820, "lon": -6.0000, "potencia_kw": 22},
    {"nombre": "AP Macarena",                                       "lat": 37.4020, "lon": -5.9920, "potencia_kw": 22},
    {"nombre": "AP Plaza de la Concordia",                          "lat": 37.3950, "lon": -5.9880, "potencia_kw": 22},
    {"nombre": "AP Magdalena - San Pablo",                          "lat": 37.3880, "lon": -5.9970, "potencia_kw": 22},
    {"nombre": "Avda. San Francisco Javier (zona azul)",            "lat": 37.3800, "lon": -5.9700, "potencia_kw": 3.7},
    {"nombre": "Avda. Reyes Catolicos (zona azul)",                 "lat": 37.3900, "lon": -6.0020, "potencia_kw": 3.7},
    {"nombre": "Teatro Lope de Vega (Avda. Maria Luisa)",           "lat": 37.3760, "lon": -5.9880, "potencia_kw": 22},
    {"nombre": "Hotel Inglaterra (Plaza Nueva)",                    "lat": 37.3900, "lon": -5.9950, "potencia_kw": 22},
    {"nombre": "Hotel Sevilla Center",                              "lat": 37.3830, "lon": -5.9750, "potencia_kw": 22},
    {"nombre": "Hotel ABBA Triana",                                 "lat": 37.3870, "lon": -6.0050, "potencia_kw": 22},
    {"nombre": "Hotel Ribera de Triana",                            "lat": 37.3830, "lon": -6.0080, "potencia_kw": 22},
    {"nombre": "C.C. Nervion Plaza",                                "lat": 37.3870, "lon": -5.9750, "potencia_kw": 50},
    {"nombre": "C.C. Plaza de Armas",                                "lat": 37.3920, "lon": -6.0030, "potencia_kw": 50},
    {"nombre": "C.C. El Mirador de Santa Justa",                    "lat": 37.3930, "lon": -5.9680, "potencia_kw": 50},
    {"nombre": "C.C. Los Arcos",                                    "lat": 37.3950, "lon": -5.9550, "potencia_kw": 50},
    {"nombre": "Aparcamiento INSUR - Mirador Santa Justa",          "lat": 37.3935, "lon": -5.9670, "potencia_kw": 22},
    {"nombre": "Aparcamiento INSUR (C/ Diego Martinez Barrios)",    "lat": 37.3925, "lon": -5.9695, "potencia_kw": 22},
    {"nombre": "Parking Sagrado Corazon",                           "lat": 37.3870, "lon": -5.9820, "potencia_kw": 22},
    {"nombre": "Parking Cristina (Puerta de Jerez)",                "lat": 37.3820, "lon": -5.9930, "potencia_kw": 22},
    {"nombre": "Carrefour San Pablo",                               "lat": 37.3980, "lon": -5.9550, "potencia_kw": 50},
    {"nombre": "REE Cartuja (C/ Inca Garcilaso)",                   "lat": 37.4200, "lon": -6.0020, "potencia_kw": 22},
    {"nombre": "El Corte Ingles San Juan de Aznalfarache",          "lat": 37.3720, "lon": -6.0330, "potencia_kw": 22},
]

print(f"Dataset de respaldo cargado: {len(ESTACIONES_FALLBACK)} estaciones")

---
✅ **Configuración cargada.** Este notebook no produce ningún efecto sobre ficheros externos - solo define constantes en memoria.